# **OPTIMIZE **

## Question: Where are the high spending customers from for targeted marketing?  

Measure + By: Spending tier by Country


In [0]:
%sql
--- Spending tier by Country

WITH customer_spending AS (
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        c.Country,
        SUM(il.UnitPrice * il.Quantity) AS total_spending
    FROM silver_invoiceline il
    JOIN silver_invoice i ON il.InvoiceId = i.InvoiceId
    JOIN silver_customer c ON i.CustomerId = c.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName, c.Country
),
percentile_thresholds AS (
    SELECT
        PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY total_spending) AS high_threshold,
        PERCENTILE_CONT(0.60) WITHIN GROUP (ORDER BY total_spending) AS medium_threshold
    FROM customer_spending
),
customer_tiers AS (
    SELECT
        cs.CustomerId,
        cs.FirstName,
        cs.LastName,
        cs.Country,
        cs.total_spending,
        CASE
            WHEN cs.total_spending >= pt.high_threshold THEN 'High'
            WHEN cs.total_spending >= pt.medium_threshold THEN 'Medium'
            ELSE 'Low'
        END AS spending_tier
    FROM customer_spending cs
    CROSS JOIN percentile_thresholds pt
)
SELECT
    Country,
    spending_tier,
    COUNT(*) AS customer_count,
    ROUND(SUM(total_spending), 2) AS total_revenue,
    ROUND(AVG(total_spending), 2) AS avg_spending,
    ROUND(MIN(total_spending), 2) AS min_spending,
    ROUND(MAX(total_spending), 2) AS max_spending
FROM customer_tiers
WHERE spending_tier = 'High'
GROUP BY Country, spending_tier
ORDER BY
    total_revenue DESC,
    Country,
    CASE spending_tier
        WHEN 'High' THEN 1
        WHEN 'Medium' THEN 2
        WHEN 'Low' THEN 3
    END


In [0]:
%sql

-- Validation 1: Source Table Row Counts and Completeness
-- Ensures all source tables have data
SELECT 'Row Count Check' AS validation_type,
       'silver_customer' AS table_name,
       COUNT(*) AS row_count,
       COUNT(CustomerId) AS non_null_pk,
       CASE WHEN COUNT(*) > 0 AND COUNT(*) = COUNT(CustomerId) THEN '✓ PASS' ELSE '✗ FAIL' END AS status
FROM silver_customer

UNION ALL

SELECT 'Row Count Check',
       'silver_invoice',
       COUNT(*),
       COUNT(InvoiceId),
       CASE WHEN COUNT(*) > 0 AND COUNT(*) = COUNT(InvoiceId) THEN '✓ PASS' ELSE '✗ FAIL' END
FROM silver_invoice

UNION ALL

SELECT 'Row Count Check',
       'silver_invoiceline',
       COUNT(*),
       COUNT(InvoiceLineId),
       CASE WHEN COUNT(*) > 0 AND COUNT(*) = COUNT(InvoiceLineId) THEN '✓ PASS' ELSE '✗ FAIL' END
FROM silver_invoiceline;


-- Validation 2: Null Values Check on Critical Columns
-- Ensures required columns have no NULLs
SELECT 'Null Check' AS validation_type,
       'silver_customer.Country' AS column_name,
       COUNT(*) AS total_rows,
       COUNT(*) - COUNT(Country) AS null_count,
       CASE WHEN COUNT(*) = COUNT(Country) THEN '✓ PASS' ELSE '✗ FAIL' END AS status
FROM silver_customer

UNION ALL

SELECT 'Null Check',
       'silver_invoiceline.UnitPrice',
       COUNT(*),
       COUNT(*) - COUNT(UnitPrice),
       CASE WHEN COUNT(*) = COUNT(UnitPrice) THEN '✓ PASS' ELSE '✗ FAIL' END
FROM silver_invoiceline

UNION ALL

SELECT 'Null Check',
       'silver_invoiceline.Quantity',
       COUNT(*),
       COUNT(*) - COUNT(Quantity),
       CASE WHEN COUNT(*) = COUNT(Quantity) THEN '✓ PASS' ELSE '✗ FAIL' END
FROM silver_invoiceline;


-- Validation 3: Business Logic Validation - Spending Calculation
-- Verifies spending calculation accuracy
WITH manual_check AS (
    SELECT
        c.CustomerId,
        SUM(il.UnitPrice * il.Quantity) AS calculated_spending
    FROM silver_invoiceline il
    JOIN silver_invoice i ON il.InvoiceId = i.InvoiceId
    JOIN silver_customer c ON i.CustomerId = c.CustomerId
    GROUP BY c.CustomerId
)
SELECT 'Business Logic' AS validation_type,
       'Spending Calculation' AS rule,
       COUNT(*) AS total_customers,
       SUM(CASE WHEN calculated_spending IS NULL THEN 1 ELSE 0 END) AS null_spending_count,
       SUM(CASE WHEN calculated_spending < 0 THEN 1 ELSE 0 END) AS negative_spending_count,
       CASE 
           WHEN SUM(CASE WHEN calculated_spending IS NULL OR calculated_spending < 0 THEN 1 ELSE 0 END) = 0 
           THEN '✓ PASS' 
           ELSE '✗ FAIL' 
       END AS status
FROM manual_check;


-- Validation 4: Percentile Threshold Validation
-- Ensures percentile thresholds are reasonable and properly ordered
WITH customer_spending AS (
    SELECT
        SUM(il.UnitPrice * il.Quantity) AS total_spending
    FROM silver_invoiceline il
    JOIN silver_invoice i ON il.InvoiceId = i.InvoiceId
    JOIN silver_customer c ON i.CustomerId = c.CustomerId
    GROUP BY c.CustomerId
),
percentile_thresholds AS (
    SELECT
        PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY total_spending) AS high_threshold,
        PERCENTILE_CONT(0.60) WITHIN GROUP (ORDER BY total_spending) AS medium_threshold,
        MIN(total_spending) AS min_spending,
        MAX(total_spending) AS max_spending
    FROM customer_spending
)
SELECT 'Percentile Validation' AS validation_type,
       'Threshold Ordering' AS rule,
       ROUND(high_threshold, 2) AS high_threshold,
       ROUND(medium_threshold, 2) AS medium_threshold,
       ROUND(min_spending, 2) AS min_spending,
       ROUND(max_spending, 2) AS max_spending,
       CASE 
           WHEN high_threshold > medium_threshold 
           AND medium_threshold > min_spending 
           AND high_threshold < max_spending
           THEN '✓ PASS' 
           ELSE '✗ FAIL' 
       END AS status
FROM percentile_thresholds;


-- Validation 5: Country Coverage Check
-- Ensures all countries with customers are represented
SELECT 'Country Coverage' AS validation_type,
       COUNT(DISTINCT c.Country) AS countries_in_source,
       COUNT(DISTINCT ct.Country) AS countries_in_result,
       COUNT(DISTINCT c.Country) - COUNT(DISTINCT ct.Country) AS missing_countries,
       CASE 
           WHEN COUNT(DISTINCT c.Country) = COUNT(DISTINCT ct.Country) THEN '✓ PASS' 
           ELSE '⚠ CHECK' 
       END AS status
FROM silver_customer c
LEFT JOIN (
    SELECT DISTINCT cs.Country
    FROM (
        SELECT
            c.CustomerId,
            c.Country,
            SUM(il.UnitPrice * il.Quantity) AS total_spending
        FROM silver_invoiceline il
        JOIN silver_invoice i ON il.InvoiceId = i.InvoiceId
        JOIN silver_customer c ON i.CustomerId = c.CustomerId
        GROUP BY c.CustomerId, c.Country
    ) cs
    CROSS JOIN (
        SELECT
            PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY total_spending) AS high_threshold
        FROM (
            SELECT SUM(il.UnitPrice * il.Quantity) AS total_spending
            FROM silver_invoiceline il
            JOIN silver_invoice i ON il.InvoiceId = i.InvoiceId
            JOIN silver_customer c ON i.CustomerId = c.CustomerId
            GROUP BY c.CustomerId
        )
    ) pt
    WHERE cs.total_spending >= pt.high_threshold
) ct ON c.Country = ct.Country;